In [1]:
!pip install requests
!pip install bs4

#импортируем необходимые библиотеки
import requests
from bs4 import BeautifulSoup
from time import sleep
from tqdm import tqdm

In [2]:
#создаём цикл для перехода по всем страницам сайта, страниц 3
urls = []
for i in range(1, 4):
    urls.append(f'https://www.culture.ru/literature/books/author-aleksandr-pushkin?sort=-views&page={i}')
urls

['https://www.culture.ru/literature/books/author-aleksandr-pushkin?sort=-views&page=1',
 'https://www.culture.ru/literature/books/author-aleksandr-pushkin?sort=-views&page=2',
 'https://www.culture.ru/literature/books/author-aleksandr-pushkin?sort=-views&page=3']

In [3]:
#создаём список, в который будут добавляться все произведения под авторством Пушкина
pushkin = []

for u in tqdm(urls):
    res = requests.get(u)
    soup = BeautifulSoup(res.text, 'html')
    links = soup.find_all('a') #находим все ссылки на странице

    for link in links:
      author = link.find('div', class_='styles_BookCard__Author__nNDt0')
      if author and author.text.strip() == "Александр Пушкин": #проверяем что автор Пушкин
          link0 = link.get('href') #извлекаем относительную ссылку
          full_link = 'https://www.culture.ru' + link0 #формируем полную ссылку
          pushkin.append(full_link)
    sleep(3)

print(len(pushkin))

100%|█████████████████████████████████████████████| 3/3 [00:16<00:00,  5.51s/it]

63


In [4]:
#из-за предложенных книг, появились дубликаты, удалим их
pushkin_1 = list(set(pushkin))
len(pushkin)

63

In [ ]:
#создаем список для информации по каждой книге
info = []

#извлекаем информацию по html-тегам
for book in tqdm(pushkin_1):
  page = requests.get(book)
  soup = BeautifulSoup(page.text, 'html')
  sleep(1)
  name = soup.find('h1', class_='styles_AboutBook__Title__9Z6mB').text
  year = soup.find('div', string='Год издания:') #находим тег, в котором находится год
  year = year.find_next('div').text if year else 'нет' #извлекаем само значение года
  ISBN = soup.find('div', string='ISBN:') #находим тег, в котором находится ISBN
  ISBN = ISBN.find_next('div').text if ISBN else 'нет' #извлекаем само значени
  description = soup.find('div', class_='styles_body__WEo9w').text
  info.append((name, year, ISBN, description))

info

 89%|██████████████████████████████████████▍    | 51/57 [01:26<00:10,  1.74s/it]

In [ ]:
#создаем датафремй, где будем хранить информацию
df = pd.DataFrame(info, columns=['Название книги', 'Год издания', 'ISBN', 'Описание книги'])
df

In [ ]:
#переформатируем датафрейм в формат эксель (файл с названием 'pushkin_books')
df.to_excel('pushkin_books.xlsx', index=False)